# R vs Python: `did` vs `csdid` comparison

**Requirement:** Run `test_results_R.R` in RStudio first to generate the CSVs in `data/`.

Compares att_gt and aggte between R and Python side by side.

In [ ]:
import os, sys, copy, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.6f}".format)

PROJECT_DIR = os.path.abspath(".")
sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "tests"))
DATA_DIR = os.path.join(PROJECT_DIR, "data")

from csdid.att_gt import ATTgt

mpdta    = pd.read_csv(os.path.join(DATA_DIR, "mpdta.csv"))
sim_data = pd.read_csv(os.path.join(DATA_DIR, "sim_data.csv"))
r_attgt  = pd.read_csv(os.path.join(DATA_DIR, "r_results_attgt.csv"))
r_aggte  = pd.read_csv(os.path.join(DATA_DIR, "r_results_aggte.csv"))

print(f"mpdta:    {len(mpdta)} rows | groups: {sorted(mpdta['first.treat'].unique())}")
print(f"sim_data: {len(sim_data)} rows | groups: {sorted(sim_data['G'].unique())}")
print(f"R att_gt: {len(r_attgt)} rows | R aggte: {len(r_aggte)} rows")

In [ ]:
import io, contextlib

SUMMARY = []

def fit(data, yname, tname, idname, gname, est_method="dr",
        xformla=None, panel=True, control_group="nevertreated",
        base_period="varying"):
    kw = dict(yname=yname, tname=tname, idname=idname, gname=gname,
              data=data, panel=panel, control_group=control_group)
    if xformla:
        kw["xformla"] = xformla
    return ATTgt(**kw).fit(est_method=est_method, bstrap=False, base_period=base_period)


def filter_to_r(res, test_id):
    """Filter the Python result to keep ONLY the (group, t) pairs that R has."""
    r_sub = r_attgt[r_attgt["test_id"] == test_id]
    if r_sub.empty:
        return res
    r_pairs = set(zip(r_sub["group"].astype(int), r_sub["t"].astype(int)))
    r_groups = set(r_sub["group"].astype(int).unique())

    res2 = copy.deepcopy(res)
    g = np.asarray(res2.MP["group"], dtype=int)
    t = np.asarray(res2.MP["t"], dtype=int)
    mask = np.array([(gg, tt) in r_pairs for gg, tt in zip(g, t)])

    res2.MP["group"] = list(g[mask])
    res2.MP["att"]   = list(np.asarray(res2.MP["att"])[mask])
    res2.MP["t"]     = list(t[mask])
    inf = res2.MP.get("inffunc")
    if isinstance(inf, dict) and "inffunc" in inf:
        res2.MP["inffunc"] = {"inffunc": np.asarray(inf["inffunc"])[:, mask]}

    old_glist = np.asarray(res2.MP["DIDparams"]["glist"])
    res2.MP["DIDparams"]["glist"] = old_glist[np.isin(old_glist, list(r_groups))]
    res2.MP["DIDparams"]["nG"] = len(res2.MP["DIDparams"]["glist"])

    for k in list(res2.results.keys()):
        v = res2.results[k]
        if isinstance(v, (list, np.ndarray)) and len(v) == len(g):
            res2.results[k] = list(np.asarray(v)[mask])
    return res2


def compare_attgt(test_id, res):
    """Compare att_gt: table with att and se from R and Python."""
    res_f = filter_to_r(res, test_id)
    py = pd.DataFrame({"group": res_f.MP["group"], "t": res_f.MP["t"],
                       "att_Py": np.asarray(res_f.MP["att"], dtype=float),
                       "se_Py": np.asarray(res_f.results["se"], dtype=float)})
    r = r_attgt[r_attgt["test_id"]==test_id][["group","t","att","se"]].rename(
        columns={"att":"att_R", "se":"se_R"})
    m = pd.merge(r, py, on=["group","t"]).sort_values(["group","t"]).reset_index(drop=True)
    m["att_diff"] = m["att_Py"] - m["att_R"]
    m["se_diff"] = m["se_Py"] - m["se_R"]
    max_att = m["att_diff"].abs().max()
    max_se = m["se_diff"].abs().max()
    st = "MATCH" if max_att < 0.01 else ("CLOSE" if max_att < 0.1 else "DIFFERS")
    SUMMARY.append({"test_id": test_id, "type": "att_gt",
                     "max_att_diff": max_att, "max_se_diff": max_se, "status": st})
    print(f"\n--- {test_id} --- [{st}]  max|att_diff|={max_att:.8f}  max|se_diff|={max_se:.8f}")
    print(m.to_string(index=False, float_format="{:.6f}".format))
    return res_f


def compare_aggte(test_id, res_filtered, typec):
    """Compare aggte: overall + detail by egt, silencing internal prints."""
    obj = copy.deepcopy(res_filtered)
    with contextlib.redirect_stdout(io.StringIO()):
        obj.aggte(typec=typec)
    a = obj.atte
    py_att, py_se = a["overall_att"], a.get("overall_se", float("nan"))
    rs = r_aggte[(r_aggte["test_id"]==test_id) & (r_aggte["aggte_type"]==typec)]
    if rs.empty:
        print(f"  aggte({typec}): Py att={py_att:+.6f} se={py_se:.6f} | R: no data")
        return
    r_att, r_se = rs["overall_att"].iloc[0], rs["overall_se"].iloc[0]
    att_d, se_d = py_att - r_att, py_se - r_se
    st = "MATCH" if abs(att_d) < 0.01 else ("CLOSE" if abs(att_d) < 0.1 else "DIFFERS")
    SUMMARY.append({"test_id": f"{test_id}/{typec}", "type": "aggte",
                     "max_att_diff": abs(att_d), "max_se_diff": abs(se_d), "status": st})

    # Overall line
    print(f"  aggte({typec:8s}) [{st}]  "
          f"att: R={r_att:+.6f} Py={py_att:+.6f} diff={att_d:+.9f}  |  "
          f"se: R={r_se:.6f} Py={py_se:.6f} diff={se_d:+.9f}")

    # Detail by egt
    if a.get("egt") is not None and not rs["egt"].isna().all():
        rd = rs[["egt","att_egt","se_egt"]].rename(columns={"att_egt":"att_R","se_egt":"se_R"})
        pd2 = pd.DataFrame({"egt": np.asarray(a["egt"]),
                             "att_Py": np.asarray(a["att_egt"], dtype=float),
                             "se_Py": np.asarray(a["se_egt"], dtype=float)})
        det = pd.merge(rd, pd2, on="egt").sort_values("egt").reset_index(drop=True)
        det["att_diff"] = det["att_Py"] - det["att_R"]
        det["se_diff"] = det["se_Py"] - det["se_R"]
        print(det.to_string(index=False, float_format="{:.6f}".format))

print("OK")

## 1. MPDTA: att_gt (DR, REG, IPW)

In [ ]:
res_mpdta = {}
for em in ["dr", "reg", "ipw"]:
    res = fit(mpdta, "lemp", "year", "countyreal", "first.treat", est_method=em)
    res_mpdta[em] = compare_attgt(f"mpdta_{em}", res)

## 2. MPDTA: aggte (DR, nevertreated)

In [ ]:
for t in ["simple", "dynamic", "group", "calendar"]:
    compare_aggte("mpdta_dr", res_mpdta["dr"], t)

## 3. MPDTA: notyettreated + aggte

In [ ]:
res_nyt = fit(mpdta, "lemp", "year", "countyreal", "first.treat",
              est_method="dr", control_group="notyettreated")
res_nyt_f = compare_attgt("mpdta_dr_nyt", res_nyt)
print()
for t in ["simple", "dynamic", "group", "calendar"]:
    compare_aggte("mpdta_dr_nyt", res_nyt_f, t)

## 4. SIM_DATA: att_gt WITH covariates (DR, REG, IPW)

In [ ]:
res_sim_cov = {}
for em in ["dr", "reg", "ipw"]:
    res = fit(sim_data, "Y", "period", "id", "G", est_method=em, xformla="Y~X")
    res_sim_cov[em] = compare_attgt(f"sim_{em}_cov", res)

## 5. SIM_DATA: att_gt WITHOUT covariates (DR, REG, IPW)

In [ ]:
res_sim_nocov = {}
for em in ["dr", "reg", "ipw"]:
    res = fit(sim_data, "Y", "period", "id", "G", est_method=em)
    res_sim_nocov[em] = compare_attgt(f"sim_{em}_nocov", res)

## 6. SIM_DATA: Repeated Cross Sections (DR, REG, IPW)

In [ ]:
res_sim_rc = {}
for em in ["dr", "reg", "ipw"]:
    res = fit(sim_data, "Y", "period", "id", "G", est_method=em, xformla="Y~X", panel=False)
    res_sim_rc[em] = compare_attgt(f"sim_{em}_rc", res)

## 7. SIM_DATA: notyettreated (panel + RC)

In [ ]:
res = fit(sim_data, "Y", "period", "id", "G", est_method="dr",
          xformla="Y~X", control_group="notyettreated")
res_nyt_p = compare_attgt("sim_dr_nyt_panel", res)

res = fit(sim_data, "Y", "period", "id", "G", est_method="dr",
          xformla="Y~X", panel=False, control_group="notyettreated")
res_nyt_r = compare_attgt("sim_dr_nyt_rc", res)

## 8. SIM_DATA: aggte (DR+cov, REG+cov, IPW+nocov, nyt)

In [ ]:
print("=== DR + covariates ===")
for t in ["simple", "dynamic", "group", "calendar"]:
    compare_aggte("sim_dr_cov", res_sim_cov["dr"], t)

print("\n=== REG + covariates ===")
for t in ["simple", "dynamic", "group", "calendar"]:
    compare_aggte("sim_reg_cov", res_sim_cov["reg"], t)

print("\n=== IPW + without covariates ===")
for t in ["simple", "dynamic", "group", "calendar"]:
    compare_aggte("sim_ipw_nocov", res_sim_nocov["ipw"], t)

print("\n=== DR + notyettreated ===")
for t in ["simple", "dynamic", "group", "calendar"]:
    compare_aggte("sim_dr_nyt_panel", res_nyt_p, t)

## 9. SIM_DATA: Universal base period

In [ ]:
res = fit(sim_data, "Y", "period", "id", "G", est_method="dr",
          xformla="Y~X", base_period="universal")
res_univ = compare_attgt("sim_dr_universal", res)
print()
for t in ["simple", "dynamic"]:
    compare_aggte("sim_dr_universal", res_univ, t)

---
## FINAL SUMMARY

In [ ]:
df = pd.DataFrame(SUMMARY)
print(df.to_string(index=False, float_format="{:.6f}".format))

print("\n" + "="*60)
total = len(df)
n_match = (df["status"]=="MATCH").sum()
n_close = (df["status"]=="CLOSE").sum()
n_diff  = (df["status"]=="DIFFERS").sum()

print(f"Total tests:             {total}")
print(f"  MATCH  (diff < 0.01):  {n_match}")
print(f"  CLOSE  (diff < 0.1):   {n_close}")
print(f"  DIFFERS (diff >= 0.1): {n_diff}")

for tp in ["att_gt", "aggte"]:
    sub = df[df["type"]==tp]
    print(f"\n  {tp}: {len(sub)} tests -> "
          f"MATCH={(sub['status']=='MATCH').sum()} "
          f"CLOSE={(sub['status']=='CLOSE').sum()} "
          f"DIFFERS={(sub['status']=='DIFFERS').sum()}")
    print(f"    max|att_diff| = {sub['max_att_diff'].max():.8f}")
    print(f"    max|se_diff|  = {sub['max_se_diff'].max():.8f}")
print("="*60)